# Day 2: Data Cleaning & SQLite DB Loading

**Bluestock Fintech – Mutual Fund Analytics**

Objective: Clean 10 raw CSVs, load into SQLite star schema, run 10 analytical queries.

## 1. Setup

In [ ]:
import os, pandas as pd, numpy as np
from sqlalchemy import create_engine, text

BASE = os.path.abspath("..")
RAW  = os.path.join(BASE, "data", "raw")
PROC = os.path.join(BASE, "data", "processed")
DB   = os.path.join(BASE, "database", "bluestock_mf.db")
print(f"Base: {BASE}
DB: {DB}")

## 2. Raw Data Overview

In [ ]:
for f in sorted(os.listdir(RAW)):
    df = pd.read_csv(os.path.join(RAW, f))
    print(f"{f:<35} {len(df):>5} rows  {list(df.columns)}")

## 3. Clean Data

In [ ]:
# Run cleaning pipeline
import subprocess, sys
result = subprocess.run([sys.executable, os.path.join(BASE, "scripts", "clean_data.py")],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0: print("STDERR:", result.stderr)

## 4. Processed Data

In [ ]:
for f in sorted(os.listdir(PROC)):
    df = pd.read_csv(os.path.join(PROC, f))
    print(f"{f:<35} {len(df):>5} rows")

## 5. Load into SQLite

In [ ]:
result = subprocess.run([sys.executable, os.path.join(BASE, "scripts", "create_schema.py")],
                        capture_output=True, text=True)
print(result.stdout)
result2 = subprocess.run([sys.executable, os.path.join(BASE, "scripts", "load_sqlite.py")],
                        capture_output=True, text=True)
print(result2.stdout)

## 6. Validate

In [ ]:
result = subprocess.run([sys.executable, os.path.join(BASE, "scripts", "validate.py")],
                        capture_output=True, text=True)
print(result.stdout)

## 7. Analytical SQL Queries

In [ ]:
engine = create_engine(f"sqlite:///{DB}")

def q(title, sql):
    print(f"
=== {title} ===")
    with engine.connect() as conn:
        df = pd.read_sql(text(sql), conn)
    print(df.to_string(index=False))
    return df

# Q1 Top 5 by AUM
q("Top 5 Funds by AUM",
  "SELECT fa.scheme_name, df.fund_house, fa.aum_cr FROM fact_aum fa JOIN dim_fund df ON fa.fund_id=df.rowid ORDER BY fa.aum_cr DESC LIMIT 5")

In [ ]:
# Q2 Avg NAV per month
q("Avg NAV per Month",
  "SELECT dd.year,dd.month,dd.month_name,ROUND(AVG(fn.nav),2) avg_nav FROM fact_nav fn JOIN dim_date dd ON fn.date_id=dd.rowid WHERE dd.is_weekend=0 GROUP BY dd.year,dd.month ORDER BY dd.year,dd.month")

In [ ]:
# Q3 SIP YoY Growth
q("SIP YoY Growth",
  "SELECT dd.year,ROUND(SUM(ft.amount),2) total_sip,COUNT(*) sip_count,ROUND(100.0*(SUM(ft.amount)-LAG(SUM(ft.amount))OVER(ORDER BY dd.year))/NULLIF(LAG(SUM(ft.amount))OVER(ORDER BY dd.year),0),2) yoy_pct FROM fact_transactions ft JOIN dim_date dd ON ft.date_id=dd.rowid WHERE ft.transaction_type='SIP' GROUP BY dd.year ORDER BY dd.year")

In [ ]:
# Q4 Transactions by State
q("Transactions by State",
  "SELECT state,COUNT(*) total,ROUND(SUM(amount),2) total_amount,COUNT(DISTINCT investor_id) investors FROM fact_transactions GROUP BY state ORDER BY total_amount DESC")

In [ ]:
# Q5 Funds ER < 1%
q("Funds with Expense Ratio < 1%",
  "SELECT scheme_name,fund_house,expense_ratio_direct FROM dim_fund WHERE expense_ratio_direct < 1.0 ORDER BY expense_ratio_direct")

In [ ]:
# Q8 KYC Status
q("KYC Status Distribution",
  "SELECT kyc_status,COUNT(DISTINCT investor_id) investors,COUNT(*) txns,ROUND(100.0*COUNT(*)/SUM(COUNT(*))OVER(),2) pct FROM fact_transactions GROUP BY kyc_status ORDER BY txns DESC")

## 8. Summary

✅ **10 CSVs cleaned** → 
✅ **bluestock_mf.db** loaded with 10 tables
✅ **12/12** validation checks pass
✅ **10 analytical queries** written in 
✅ **Data dictionary** in 